In [2]:
import os, json, time, math, random
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support, confusion_matrix)

SEED = 2026
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
seed_all()

MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"
MAX_LEN, BATCH = 128, 64
LR_ENCODER, LR_HEAD = 2e-5, 1e-3
WEIGHT_DECAY, WARMUP, CLIP, LAMBDA_BAND = 0.01, 0.06, 1.0, 1.0
USE_AMP = True

THRESHOLD = 0.90
EVAL_EVERY, PATIENCE = 1000, 3

REC, BAND = ["recommended", "not_recommended", "no_idea"], ["low", "medium", "high"]
REC_ID = {c: i for i, c in enumerate(REC)}; BAND_ID = {c: i for i, c in enumerate(BAND)}

def find(name):
    for root in ["/kaggle/working", "/kaggle/input"]:
        hits = sorted(Path(root).rglob(name))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found in /kaggle/working or /kaggle/input")

CKPT, METRICS03 = find("transformer_best.pt"), find("transformer_metrics.json")
OUT = Path("/kaggle/working"); dev = torch.device("cuda")
print("Checkpoint:", CKPT)
print("Metrics:", METRICS03)

Checkpoint: /kaggle/input/notebooks/moanaz/03-transfer-learning-finetuning/transformer_best.pt
Metrics: /kaggle/input/notebooks/moanaz/03-transfer-learning-finetuning/transformer_metrics.json


In [8]:
def load(name): return pd.read_csv(find(name), dtype=str, keep_default_na=False, na_values=[""])
train, val, test, pseudo = load("train_sub.csv"), load("val_sub.csv"), load("test.csv"), load("pseudo_sub.csv")
assert not (set(train["id"]) & set(pseudo["id"])), "pseudo pool overlaps train!"
print(len(train), len(val), len(test), len(pseudo))

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, ids, yr, yb): self.ids, self.yr, self.yb = ids, yr, yb
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.ids[i], self.yr[i], self.yb[i]

def encode(df):
    texts, ids = df["body"].tolist(), []
    for i in range(0, len(texts), 20_000):
        ids += tok(texts[i:i+20_000], truncation=True, max_length=MAX_LEN)["input_ids"]
    n = len(df)
    yr = (df["recommendation_status"].map(REC_ID).fillna(-100).astype(int).values
          if "recommendation_status" in df else np.full(n, -100))
    yb = (df["rating_band"].map(BAND_ID).fillna(-100).astype(int).values
          if "rating_band" in df else np.full(n, -100))
    return DS(ids, yr, yb)

def concat(a, b): return DS(a.ids + b.ids, np.concatenate([a.yr, b.yr]), np.concatenate([a.yb, b.yb]))

def collate(batch):
    L = max(len(b[0]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for k, (x, _, _) in enumerate(batch):
        ids[k, :len(x)] = torch.tensor(x); att[k, :len(x)] = 1
    return ids, att, torch.tensor([b[1] for b in batch]), torch.tensor([b[2] for b in batch])

t0 = time.time()
ds_tr, ds_va, ds_te, ds_ps = encode(train), encode(val), encode(test), encode(pseudo)
print(f"tokenized in {time.time()-t0:.0f}s")

354599 92164 917785 74090
tokenized in 78s


In [13]:
class TwoHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(MODEL_NAME)
        h = self.enc.config.hidden_size
        self.drop, self.rec, self.band = nn.Dropout(0.1), nn.Linear(h, 3), nn.Linear(h, 3)
    def forward(self, ids, att):
        z = self.drop(self.enc(input_ids=ids, attention_mask=att).last_hidden_state[:, 0])
        return self.rec(z), self.band(z)

def new_model():
    m = TwoHead()
    m.load_state_dict(torch.load(CKPT, map_location="cpu"))
    return m.to(dev)

def class_weights(y, k):
    cnt = np.bincount(y[y >= 0], minlength=k).astype(float)
    return torch.tensor(cnt.sum() / (k * cnt), dtype=torch.float32, device=dev)
W_REC, W_BAND = class_weights(ds_tr.yr, 3), class_weights(ds_tr.yb, 3)

def masked_ce(logits, y, w):
    ok = y != -100
    if not ok.any(): return logits.sum() * 0.0
    return F.cross_entropy(logits[ok], y[ok], weight=w)

def evaluate(y_true, y_pred, classes, ordinal=False):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    labels = list(range(len(classes)))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    out = {"n": int(len(y_true)), "accuracy": float(accuracy_score(y_true, y_pred)),
           "macro_f1": float(f.mean()), "macro_precision": float(p.mean()), "macro_recall": float(r.mean()),
           "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
           "per_class": {c: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]),
                             "support": int(s[i])} for i, c in enumerate(classes)},
           "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist()}
    if ordinal: out["ordinal_distance_error"] = float(np.abs(y_true - y_pred).mean())
    return out

@torch.no_grad()
def predict(model, ds, bs=256):
    was = model.training; model.eval()
    order = np.argsort([len(x) for x in ds.ids])
    dl = DataLoader(torch.utils.data.Subset(ds, order), batch_size=bs, shuffle=False,
                    collate_fn=collate, num_workers=2)
    Pr_, Pb_ = [], []
    for ids, att, _, _ in dl:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            a, b = model(ids.to(dev), att.to(dev))
        Pr_.append(F.softmax(a.float(), -1).cpu()); Pb_.append(F.softmax(b.float(), -1).cpu())
    Pr_, Pb_ = torch.cat(Pr_).numpy(), torch.cat(Pb_).numpy()
    inv = np.empty_like(order); inv[order] = np.arange(len(order))
    model.train(was)
    return Pr_[inv], Pb_[inv]

def score_split(model, ds):
    Pr, Pb = predict(model, ds)
    m1, m2 = ds.yr >= 0, ds.yb >= 0
    return {"rec": evaluate(ds.yr[m1], Pr[m1].argmax(1), REC),
            "band": evaluate(ds.yb[m2], Pb[m2].argmax(1), BAND, ordinal=True)}, Pr, Pb

In [14]:
sup = new_model()
Pr_ps, _ = predict(sup, ds_ps)
conf, pred = Pr_ps.max(1), Pr_ps.argmax(1)
accepted = conf >= THRESHOLD
pseudo_df = pd.DataFrame({"id": pseudo["id"].values, "pred": [REC[i] for i in pred],
                          "conf": conf, "accepted": accepted})
stats = {"threshold": THRESHOLD, "pool_rows": int(len(pseudo)), "accepted": int(accepted.sum()),
         "accepted_pct": float(accepted.mean() * 100),
         "pred_distribution_all": pseudo_df["pred"].value_counts().to_dict(),
         "accepted_distribution": pseudo_df[pseudo_df.accepted]["pred"].value_counts().to_dict(),
         "acceptance_rate_by_pred_class": pseudo_df.groupby("pred")["accepted"].mean().to_dict()}
print(json.dumps(stats, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
del sup; torch.cuda.empty_cache()

keep = np.flatnonzero(accepted)
ds_pl = DS([ds_ps.ids[i] for i in keep], pred[keep].astype(int), np.full(len(keep), -100))
ds_semi = concat(ds_tr, ds_pl)
semi_steps = math.ceil(len(ds_semi) / BATCH)
print("semi rows:", len(ds_semi), "| steps (1 epoch):", semi_steps)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

{
  "threshold": 0.9,
  "pool_rows": 74090,
  "accepted": 35496,
  "accepted_pct": 47.909299500607375,
  "pred_distribution_all": {
    "recommended": 53473,
    "no_idea": 11885,
    "not_recommended": 8732
  },
  "accepted_distribution": {
    "recommended": 32976,
    "not_recommended": 2481,
    "no_idea": 39
  },
  "acceptance_rate_by_pred_class": {
    "no_idea": 0.003281447202355911,
    "not_recommended": 0.2841273476866697,
    "recommended": 0.6166850560095749
  }
}
semi rows: 390095 | steps (1 epoch): 6096


In [16]:
def train_continue(name, ds, total_steps):
    seed_all()
    model = new_model(); model.train()
    nd = ("bias", "LayerNorm.weight")
    enc = list(model.enc.named_parameters())
    groups = [
        {"params": [p for n, p in enc if not any(x in n for x in nd)], "lr": LR_ENCODER, "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in enc if any(x in n for x in nd)], "lr": LR_ENCODER, "weight_decay": 0.0},
        {"params": list(model.rec.parameters()) + list(model.band.parameters()), "lr": LR_HEAD, "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total_steps), total_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    g = torch.Generator(); g.manual_seed(SEED)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate, generator=g,
                    num_workers=2, pin_memory=True)
    it = iter(dl)
    hist = {"step": [], "train_loss": [], "val_step": [], "val_f1_rec": [], "val_f1_band": []}
    best, bad, skipped, run, t0 = {"score": -1, "step": 0, "state": None}, 0, 0, [], time.time()
    for step in range(1, total_steps + 1):
        try: ids, att, yr, yb = next(it)
        except StopIteration: it = iter(dl); ids, att, yr, yb = next(it)
        ids, att, yr, yb = ids.to(dev), att.to(dev), yr.to(dev), yb.to(dev)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            a, b = model(ids, att)
        loss = masked_ce(a.float(), yr, W_REC) + LAMBDA_BAND * masked_ce(b.float(), yb, W_BAND)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        gn = nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        if not torch.isfinite(gn): skipped += 1
        
        old_scale = scaler.get_scale()
        scaler.step(opt); scaler.update()
        if scaler.get_scale() == old_scale: sched.step()
        
        run.append(loss.item())
        if step % 200 == 0:
            hist["step"].append(step); hist["train_loss"].append(float(np.mean(run))); run = []
            print(f"[{name}] step {step}/{total_steps} loss={hist['train_loss'][-1]:.4f} "
                  f"skipped={skipped} scale={scaler.get_scale():.0f} ({time.time()-t0:.0f}s)", flush=True)
        if step % EVAL_EVERY == 0 or step == total_steps:
            res, _, _ = score_split(model, ds_va)
            fr, fb = res["rec"]["macro_f1"], res["band"]["macro_f1"]; sc = (fr + fb) / 2
            hist["val_step"].append(step); hist["val_f1_rec"].append(fr); hist["val_f1_band"].append(fb)
            print(f"  [{name}] VAL step {step}: rec={fr:.4f} band={fb:.4f} mean={sc:.4f}", flush=True)
            if sc > best["score"]:
                best.update(score=sc, step=step, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()}); bad = 0
                torch.save(best["state"], OUT / f"{name}_best.pt")
            else:
                bad += 1
                if bad >= PATIENCE: print("  early stopping"); break
    model.load_state_dict(best["state"])
    print(f"[{name}] selected step {best['step']} score {best['score']:.4f}")
    return model, hist, best["step"]

def save_preds(tag, Pr, Pb):
    for task, P, y, classes in (("recommendation", Pr, ds_te.yr, REC), ("rating_band", Pb, ds_te.yb, BAND)):
        m = y >= 0
        d = pd.DataFrame({"id": test["id"].values[m], "y_true": [classes[i] for i in y[m]],
                          "y_pred": [classes[i] for i in P[m].argmax(1)]})
        for i, c in enumerate(classes): d[f"p_{c}"] = P[m][:, i]
        d.to_parquet(OUT / f"preds_{tag}_{task}_test.parquet", index=False)

m_semi, h_semi, step_semi = train_continue("semi", ds_semi, semi_steps)
res_semi, Pr, Pb = score_split(m_semi, ds_te); save_preds("semi", Pr, Pb)
del m_semi; torch.cuda.empty_cache()

m_ctl, h_ctl, step_ctl = train_continue("control", ds_tr, semi_steps)
res_ctl, Pr, Pb = score_split(m_ctl, ds_te); save_preds("control", Pr, Pb)
del m_ctl; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[semi] step 200/6096 loss=1.2143 skipped=3 scale=8192 (48s)
[semi] step 400/6096 loss=1.2727 skipped=3 scale=8192 (94s)
[semi] step 600/6096 loss=1.3096 skipped=3 scale=8192 (142s)
[semi] step 800/6096 loss=1.2805 skipped=3 scale=8192 (191s)
[semi] step 1000/6096 loss=1.2824 skipped=3 scale=8192 (240s)
  [semi] VAL step 1000: rec=0.6585 band=0.6369 mean=0.6477
[semi] step 1200/6096 loss=1.2758 skipped=3 scale=8192 (306s)
[semi] step 1400/6096 loss=1.2783 skipped=3 scale=8192 (355s)
[semi] step 1600/6096 loss=1.2992 skipped=3 scale=8192 (403s)
[semi] step 1800/6096 loss=1.2458 skipped=3 scale=8192 (450s)
[semi] step 2000/6096 loss=1.2891 skipped=3 scale=8192 (498s)
  [semi] VAL step 2000: rec=0.6726 band=0.6428 mean=0.6577
[semi] step 2200/6096 loss=1.2677 skipped=4 scale=8192 (564s)
[semi] step 2400/6096 loss=1.2600 skipped=4 scale=8192 (612s)
[semi] step 2600/6096 loss=1.2588 skipped=4 scale=8192 (661s)
[semi] step 2800/6096 loss=1.2457 skipped=4 scale=8192 (709s)
[semi] step 3000/609

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[control] step 200/6096 loss=1.3306 skipped=3 scale=8192 (48s)
[control] step 400/6096 loss=1.3372 skipped=3 scale=8192 (97s)
[control] step 600/6096 loss=1.3492 skipped=3 scale=8192 (145s)
[control] step 800/6096 loss=1.2995 skipped=3 scale=8192 (194s)
[control] step 1000/6096 loss=1.3153 skipped=3 scale=8192 (244s)
  [control] VAL step 1000: rec=0.6723 band=0.5789 mean=0.6256
[control] step 1200/6096 loss=1.2903 skipped=3 scale=8192 (309s)
[control] step 1400/6096 loss=1.2937 skipped=3 scale=8192 (357s)
[control] step 1600/6096 loss=1.2654 skipped=3 scale=8192 (404s)
[control] step 1800/6096 loss=1.2684 skipped=3 scale=8192 (453s)
[control] step 2000/6096 loss=1.2672 skipped=3 scale=8192 (503s)
  [control] VAL step 2000: rec=0.6950 band=0.6471 mean=0.6711
[control] step 2200/6096 loss=1.2525 skipped=4 scale=8192 (570s)
[control] step 2400/6096 loss=1.2890 skipped=4 scale=8192 (618s)
[control] step 2600/6096 loss=1.2374 skipped=4 scale=8192 (667s)
[control] step 2800/6096 loss=1.2749 

In [17]:
res_sup = json.load(open(METRICS03))["test"]            
metrics = ["accuracy", "macro_f1", "macro_precision", "macro_recall", "balanced_accuracy"]
rows = []
for head, k in (("recommendation", "rec"), ("rating_band", "band")):
    for m in metrics + (["ordinal_distance_error"] if k == "band" else []):
        a, c, s = res_sup[k][m], res_ctl[k][m], res_semi[k][m]
        rows.append({"head": head, "metric": m, "supervised": a, "control": c, "semi": s,
                     "semi-sup": s - a, "semi-control": s - c})
tbl = pd.DataFrame(rows).round(4); print(tbl.to_string(index=False))

for head, k in (("recommendation", "rec"), ("rating_band", "band")):
    print(f"\n== per-class F1 ({head})")
    for c in res_semi[k]["per_class"]:
        print(f"  {c:16s} sup={res_sup[k]['per_class'][c]['f1']:.3f} "
              f"ctl={res_ctl[k]['per_class'][c]['f1']:.3f} semi={res_semi[k]['per_class'][c]['f1']:.3f}")
tbl.to_csv(OUT / "semi_comparison.csv", index=False)

          head                 metric  supervised  control   semi  semi-sup  semi-control
recommendation               accuracy      0.8312   0.8391 0.8307   -0.0005       -0.0084
recommendation               macro_f1      0.6954   0.7053 0.7056    0.0102        0.0002
recommendation        macro_precision      0.6653   0.6810 0.6863    0.0210        0.0053
recommendation           macro_recall      0.7411   0.7417 0.7457    0.0047        0.0040
recommendation      balanced_accuracy      0.7411   0.7417 0.7457    0.0047        0.0040
   rating_band               accuracy      0.7576   0.7640 0.7541   -0.0035       -0.0099
   rating_band               macro_f1      0.6665   0.6686 0.6675    0.0010       -0.0012
   rating_band        macro_precision      0.6519   0.6544 0.6494   -0.0025       -0.0050
   rating_band           macro_recall      0.6849   0.6858 0.6917    0.0068        0.0059
   rating_band      balanced_accuracy      0.6849   0.6858 0.6917    0.0068        0.0059
   rating_

In [18]:
ref = load("pseudo_reference_sub.csv").rename(columns={"recommendation_status": "ref"})
a = pseudo_df.merge(ref, on="id", how="left"); assert a["ref"].notna().all()
acc, rej = a[a.accepted], a[~a.accepted]
audit = {"accuracy_accepted": float((acc["pred"] == acc["ref"]).mean()),
         "accuracy_rejected": float((rej["pred"] == rej["ref"]).mean()) if len(rej) else None,
         "per_class_precision_accepted": {c: float((acc[acc.pred == c]["ref"] == c).mean())
                                          for c in REC if (acc.pred == c).any()},
         "reference_distribution_pool": a["ref"].value_counts().to_dict(),
         "reference_distribution_accepted": acc["ref"].value_counts().to_dict()}
print(json.dumps(audit, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))

json.dump({"pseudo_stats": stats, "pseudo_audit": audit, "semi_steps": semi_steps,
           "selected_step": {"semi": step_semi, "control": step_ctl},
           "supervised_test": res_sup, "control_test": res_ctl, "semi_test": res_semi,
           "hist": {"semi": h_semi, "control": h_ctl}},
          open(OUT / "semi_metrics.json", "w"), indent=2,
          default=lambda o: o.item() if hasattr(o, "item") else str(o))

{
  "accuracy_accepted": 0.9800822627901735,
  "accuracy_rejected": 0.6894595014769135,
  "per_class_precision_accepted": {
    "recommended": 0.9866872877244056,
    "not_recommended": 0.8964127367996776,
    "no_idea": 0.717948717948718
  },
  "reference_distribution_pool": {
    "recommended": 58641,
    "no_idea": 8406,
    "not_recommended": 7043
  },
  "reference_distribution_accepted": {
    "recommended": 32620,
    "not_recommended": 2287,
    "no_idea": 589
  }
}


In [2]:
import numpy as np, pandas as pd
from pathlib import Path

REP = Path("../reports")

def macro_f1_from_cm(cm):
    tp = np.diag(cm).astype(float)
    p = tp / np.maximum(cm.sum(0), 1); r = tp / np.maximum(cm.sum(1), 1)
    return (2 * p * r / np.maximum(p + r, 1e-12)).mean()

def paired_bootstrap(task, classes, tagA, tagB, B=1000, seed=2026):
    a = pd.read_parquet(REP / f"preds_{tagA}_{task}_test.parquet")
    b = pd.read_parquet(REP / f"preds_{tagB}_{task}_test.parquet")
    m = a.merge(b, on="id", suffixes=("_a", "_b"))
    cid = {c: i for i, c in enumerate(classes)}
    y, pa, pb = (m[c].map(cid).values for c in ("y_true_a", "y_pred_a", "y_pred_b"))
    k, n, rng, d = len(classes), len(y), np.random.default_rng(seed), []
    for _ in range(B):
        i = rng.integers(0, n, n)
        ca = np.bincount(y[i] * k + pa[i], minlength=k * k).reshape(k, k)
        cb = np.bincount(y[i] * k + pb[i], minlength=k * k).reshape(k, k)
        d.append(macro_f1_from_cm(cb) - macro_f1_from_cm(ca))
    lo, hi = np.percentile(d, [2.5, 97.5])
    return round(float(np.mean(d)), 4), round(float(lo), 4), round(float(hi), 4)

SETS = {"recommendation": ["recommended", "not_recommended", "no_idea"],
        "rating_band": ["low", "medium", "high"]}
for task, classes in SETS.items():
    print(task, "semi vs control   (mean diff, 95% CI):", paired_bootstrap(task, classes, "control", "semi"))
    print(task, "control vs tfidf  (mean diff, 95% CI):", paired_bootstrap(task, classes, "tfidf", "control"))

recommendation semi vs control   (mean diff, 95% CI): (0.0002, -0.0007, 0.0011)
recommendation control vs tfidf  (mean diff, 95% CI): (0.0062, 0.0049, 0.0074)
rating_band semi vs control   (mean diff, 95% CI): (-0.0012, -0.0018, -0.0005)
rating_band control vs tfidf  (mean diff, 95% CI): (0.0108, 0.0098, 0.0118)


In [4]:
import re
test = pd.read_csv("../data/processed/test.csv", dtype=str, keep_default_na=False,
                   na_values=[""], usecols=["id", "body"])
REC = ["recommended", "not_recommended", "no_idea"]
tf = pd.read_parquet(REP / "preds_tfidf_recommendation_test.parquet")
tr = pd.read_parquet(REP / "preds_semi_recommendation_test.parquet")
m = tf.merge(tr, on="id", suffixes=("_tf", "_tr")).merge(test, on="id")
m["y_true"] = m["y_true_tf"]
m["tf_ok"], m["tr_ok"] = m["y_pred_tf"] == m["y_true"], m["y_pred_tr"] == m["y_true"]
m["conf_tr"] = m[[f"p_{c}_tr" for c in REC]].max(axis=1)
print("agreement:", (m.y_pred_tf == m.y_pred_tr).mean().round(3),
      "| both wrong:", ((~m.tf_ok) & (~m.tr_ok)).mean().round(3))

NEG = {"نه", "نیست", "نبود", "نداره", "ندارد", "هیچ", "بدون"}
MIX = {"ولی", "اما", "ولیکن", "هرچند"}
def tags(t):
    w = t.split(); out = []
    if len(w) <= 4: out.append("short")
    if len(w) >= 80: out.append("long")
    if any(x in NEG or x.startswith("نمی") for x in w): out.append("negation")
    if any(x in MIX for x in w): out.append("mixed")
    if "؟" in t or "?" in t: out.append("question")
    return "|".join(out)
m["tags"] = m["body"].map(tags)

def pick(mask, n, group):
    d = m[mask]; d = d.sample(min(n, len(d)), random_state=2026).copy(); d["group"] = group; return d
wrong_any = ~m.tf_ok | ~m.tr_ok
parts = [pick(~m.tf_ok & ~m.tr_ok, 10, "both_wrong"),
         pick(m.tf_ok & ~m.tr_ok, 6, "tfidf_right_transformer_wrong"),
         pick(~m.tf_ok & m.tr_ok, 6, "transformer_right_tfidf_wrong"),
         pick(~m.tr_ok & (m.conf_tr >= 0.9), 5, "confident_wrong"),
         pick(~m.tr_ok & m.tags.str.contains("negation"), 4, "negation_wrong"),
         pick(~m.tr_ok & m.tags.str.contains("mixed"), 4, "mixed_wrong"),
         pick(~m.tr_ok & m.tags.str.contains("short"), 4, "short_wrong")]
cand = pd.concat(parts).drop_duplicates("id")
cand["manual_category"] = ""
cols = ["group", "id", "body", "y_true", "y_pred_tf", "y_pred_tr", "conf_tr", "tags", "manual_category"]
cand[cols].to_csv(REP / "error_analysis_candidates.csv", index=False, encoding="utf-8-sig")
print(cand["group"].value_counts())

agreement: 0.897 | both wrong: 0.112
group
both_wrong                       10
tfidf_right_transformer_wrong     6
transformer_right_tfidf_wrong     6
confident_wrong                   5
negation_wrong                    4
mixed_wrong                       4
short_wrong                       4
Name: count, dtype: int64


In [11]:
m["dup_count"] = m.groupby("body")["id"].transform("count")
m["dup_label_conflict"] = m.groupby("body")["y_true"].transform("nunique") > 1

cand = cand.drop(columns=["dup_count", "dup_label_conflict"], errors="ignore")
cand = cand.merge(m[["id", "dup_count", "dup_label_conflict"]], on="id", how="left")

cols2 = cols[:-1] + ["dup_count", "dup_label_conflict", "manual_category"]
cand[cols2].to_csv(REP / "error_analysis_candidates.csv", index=False, encoding="utf-8-sig")

import duckdb

rate = duckdb.connect().execute(
    "SELECT id, rate FROM read_parquet('../data/processed/comments.parquet')").df()
BCODE = {"low": 0, "medium": 1, "high": 2}
bt = pd.read_parquet(REP / "preds_tfidf_rating_band_test.parquet")
bs = pd.read_parquet(REP / "preds_semi_rating_band_test.parquet")
b = bt.merge(bs, on="id", suffixes=("_tf", "_tr")).merge(test, on="id").merge(rate, on="id", how="left")
b["y_true"] = b["y_true_tf"]
b["dist_tf"] = (b["y_pred_tf"].map(BCODE) - b["y_true"].map(BCODE)).abs()
b["dist_tr"] = (b["y_pred_tr"].map(BCODE) - b["y_true"].map(BCODE)).abs()

def pickb(mask, n, g):
    d = b[mask]; d = d.sample(min(n, len(d)), random_state=2026).copy(); d["group"] = g; return d

bc = pd.concat([pickb(b.dist_tr == 1, 6, "band_adjacent_error"),
                pickb(b.dist_tr == 2, 4, "band_far_error"),
                pickb((b.dist_tf == 0) & (b.dist_tr > 0), 3, "tfidf_right_transformer_wrong"),
                pickb((b.dist_tf > 0) & (b.dist_tr == 0), 3, "transformer_right_tfidf_wrong")]).drop_duplicates("id")
bc["manual_category"] = ""

bc[["group", "id", "body", "rate", "y_true", "y_pred_tf", "y_pred_tr", "manual_category"]].to_csv(
    REP / "error_analysis_candidates_band.csv", index=False, encoding="utf-8-sig")

PermissionError: [Errno 13] Permission denied: '..\\reports\\error_analysis_candidates.csv'

In [10]:
cnt = m.groupby(["body", "y_true"]).size().rename("c").reset_index()
best = cnt.groupby("body")["c"].max().sum()        
N = len(m)
print("سقف accuracy برای هر مدل متنی (فقط از تکرارها):", round(best / N, 4))
print("حداقل خطای غیرقابل‌اجتناب:", round(1 - best / N, 4))
conf = m[m["dup_label_conflict"]]
print("سهم ردیف‌هایی که متنشون برچسب متناقض داره:", round(len(conf) / N, 4))
print(conf["y_true"].value_counts(normalize=True).round(3))

سقف accuracy برای هر مدل متنی (فقط از تکرارها): 0.976
حداقل خطای غیرقابل‌اجتناب: 0.024
سهم ردیف‌هایی که متنشون برچسب متناقض داره: 0.2253
y_true
recommended        0.842
no_idea            0.112
not_recommended    0.046
Name: proportion, dtype: float64


In [12]:
from transformers import AutoConfig

AutoConfig.from_pretrained("HooshvareLab/bert-fa-base-uncased").save_pretrained("artifacts/transformer/config")

e:\maktab seesion\home work\code\project-3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
